# L77 · 音乐特征工程 — chroma、RMS 能量、ZCR，调用 aurora.music 从零实现

**目标**：实现三个音乐特有特征——chroma向量调性（tonality）、RMS能量包络（响度曲线）、节拍检测（BPM）——全部从 numpy 手动构建，无外部库。

🔗 **Aurora 连接**：`aurora.music` 子包已在 `src/aurora/music/features.py` 中完整实现，导出了 `chroma_vector`、`rms_envelope`、`zero_crossing_rate`、`onset_envelope` 和 `beat_track`。本节练习从零手写这些函数，目的是理解其内部逻辑；完成后可与 `src/aurora/music/features.py` 的参考实现逐行对比。

一段旋律的「调性感」来自哪些频率在哪些音高类上的能量分布——这就是 chroma；「响度随时间的起伏」则由逐帧 RMS 捕获；节拍感知本质上是在能量包络里找等间距的峰。三个特征合起来，足以区分音乐流派、检测和弦变化、同步节拍，是音乐信息检索（MIR）的基础工具箱。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.stft import stft
from aurora.audio.io import read_wav

## 1. Chroma：调性指纹

STFT 给出每个 FFT bin 对应的频率 `f = k * sr / n_fft`（k = 0,1,...,n_fft//2）。把频率映射到 MIDI 音符：

```
midi = 12 * log2(f / 440) + 69
```

音高类（pitch class）= `round(midi) mod 12`，对应 C=0, C#=1, D=2, ..., B=11。把落在同一音高类的所有 bin 的功率加总，得到长度为 12 的向量，再归一化到 [0,1]。

直觉：A大调音阶（A B C# D E F# G#）演奏时，chroma 的第 0,2,4,6,9,11 位应显著高于其余。

In [ ]:
# 演示：构造一个只含 A4 (440 Hz) 的纯音，观察 chroma
sr = 22050
duration = 1.0
t = np.arange(int(sr * duration)) / sr
x_pure_a = np.sin(2 * np.pi * 440 * t)   # 纯 A4

n_fft = 2048
freqs = np.arange(n_fft // 2 + 1) * sr / n_fft  # 每个 bin 的频率

# 手动算功率谱（单帧演示）
frame = x_pure_a[:n_fft] * np.hanning(n_fft)
spectrum = np.abs(np.fft.rfft(frame)) ** 2

# 音高类映射
with np.errstate(divide='ignore', invalid='ignore'):
    midi = np.where(freqs > 0, 12 * np.log2(freqs / 440) + 69, -1)
pitch_class = (np.round(midi).astype(int)) % 12

chroma_demo = np.zeros(12)
for pc, p in zip(pitch_class, spectrum):
    if 0 <= pc < 12:
        chroma_demo[pc] += p
chroma_demo /= (chroma_demo.max() + 1e-8)

labels = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
plt.figure(figsize=(8, 3))
plt.bar(labels, chroma_demo)
plt.title('Chroma — 纯 A4 (440 Hz)；期望 A=9 最高')
plt.ylabel('归一化能量')
plt.tight_layout()
plt.show()
print(f'能量最高的音高类 index = {chroma_demo.argmax()}  (应为 9 = A)')

## 2. RMS 能量包络

对音频信号做短时分帧（帧长 `frame_len`，跳步 `hop`），对每帧计算均方根：

```
rms[i] = sqrt( mean( frame_i ** 2 ) )
```

结果是长度为 `n_frames` 的向量，描述响度随时间的变化。静音段 RMS ≈ 0，击鼓/齐奏段 RMS 可达 0.3–0.8（归一化到 ±1 的音频）。

帧数公式：`n_frames = 1 + (len(x) - frame_len) // hop`。

In [ ]:
# 演示：方波 + 静音拼接，观察 RMS 包络
sr = 22050
silence = np.zeros(sr // 2)            # 0.5 s 静音
loud = 0.8 * np.sign(np.sin(2 * np.pi * 220 * np.arange(sr) / sr))  # 方波 1s
x_demo = np.concatenate([silence, loud, silence])

frame_len, hop = 2048, 512
n_frames = 1 + (len(x_demo) - frame_len) // hop
frames = np.array([x_demo[i*hop : i*hop + frame_len] for i in range(n_frames)])
rms_demo = np.sqrt(np.mean(frames ** 2, axis=1))

times = np.arange(n_frames) * hop / sr
plt.figure(figsize=(9, 3))
plt.plot(times, rms_demo)
plt.xlabel('时间 (s)')
plt.ylabel('RMS')
plt.title('RMS 能量包络：静音→方波→静音')
plt.tight_layout()
plt.show()

## 3. 节拍检测（自相关（autocorrelation）法）

**思路**：
1. 计算 RMS 能量包络（onset strength 的粗略替代）。
2. 对包络做自相关：`ac[lag] = sum( rms[t] * rms[t+lag] )`。
3. 在对应 BPM 范围（通常 60–240 BPM）的 lag 区间内找峰值 lag*。
4. `BPM = 60 / (lag* * hop / sr)`。

```
lag_min = int(60 / 240 * sr / hop)   # 对应 240 BPM
lag_max = int(60 / 60  * sr / hop)   # 对应  60 BPM
lag_star = lag_min + ac[lag_min:lag_max].argmax()
BPM = 60 / (lag_star * hop / sr)
```

自相关在「周期性节拍」出现时会在对应 lag 产生明显峰，这是无需标注数据的无监督估计。

In [ ]:
# 演示：构造 120 BPM 的击鼓包络，验证自相关 BPM 估计
sr = 22050
hop = 512
frame_len = 2048
bpm_true = 120
beat_period_samples = int(60 / bpm_true * sr)   # 每拍的采样数

# 用衰减脉冲模拟击鼓瞬态
x_drum = np.zeros(sr * 4)   # 4 s
for beat_start in range(0, len(x_drum), beat_period_samples):
    end = min(beat_start + 2205, len(x_drum))  # 100 ms 衰减
    length = end - beat_start
    x_drum[beat_start:end] += np.exp(-np.arange(length) / 1000) * np.random.randn(length) * 0.5

# RMS 包络
n_frames = 1 + (len(x_drum) - frame_len) // hop
frames_d = np.array([x_drum[i*hop:i*hop+frame_len] for i in range(n_frames)])
rms_d = np.sqrt(np.mean(frames_d**2, axis=1))

# 自相关
ac = np.correlate(rms_d, rms_d, mode='full')
ac = ac[len(ac)//2:]   # 只要正 lag

lag_min = max(1, int(60/240 * sr / hop))
lag_max = int(60/60  * sr / hop)
lag_star = lag_min + ac[lag_min:lag_max].argmax()
bpm_est = 60 / (lag_star * hop / sr)

print(f'真实 BPM = {bpm_true},  估计 BPM = {bpm_est:.1f}')

plt.figure(figsize=(9,3))
lags = np.arange(lag_min, lag_max) * hop / sr
plt.plot(lags, ac[lag_min:lag_max])
plt.axvline(lag_star * hop / sr, color='r', linestyle='--', label=f'峰值 lag → {bpm_est:.1f} BPM')
plt.xlabel('Lag (s)')
plt.ylabel('自相关')
plt.title('RMS 包络自相关 — 峰值对应节拍周期')
plt.legend()
plt.tight_layout()
plt.show()

## 4. ✏️ 实现 `chroma_vector(power_spectrum, sr, n_fft)`

**签名**：`chroma_vector(power_spectrum, sr, n_fft) -> np.ndarray  # shape (12,)`

**推理路线**：
1. 计算每个 FFT bin 对应的频率：`freqs[k] = k * sr / n_fft`（k = 0,...,n_fft//2）
2. 将频率转为 MIDI 音符：`midi = 12 * log2(freq / 440) + 69`（freq=0 时跳过）
3. 音高类 = `round(midi) mod 12`，把同一音高类的功率累加到 `chroma[pc]`
4. 除以最大值（加小量防零除）归一化到 [0, 1]

> ⚠️ **归一化约定说明**：本练习使用 **max-norm**（除以最大值），输出范围 [0, 1]，测试断言 `ch[9] ≈ 1.0`。
> `src/aurora/music/features.py` 中的参考实现使用 **L1-norm**（除以总和），输出为概率分布（各元素之和=1）。
> 两种归一化都合法；本节采用 max-norm 是为了让单音测试直观（最高音高类 = 1.0）。完成练习后可尝试改为 L1-norm 并对比效果。

**参考输入输出**：
- 输入：`power_spectrum` 在 bin `k = round(440 * n_fft / sr)` 处有尖峰，其余近零
- 输出：`chroma[9] ≈ 1.0`（A=9），其余 ≈ 0

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def chroma_vector(power_spectrum, sr, n_fft):
    """
    将功率谱映射到 12 维 chroma 向量。

    Parameters
    ----------
    power_spectrum : np.ndarray, shape (n_fft//2 + 1,)
    sr : int  采样率
    n_fft : int  FFT 点数

    Returns
    -------
    chroma : np.ndarray, shape (12,)  归一化到 [0, 1]
    """
    n_bins = len(power_spectrum)
    freqs = np.arange(n_bins) * sr / n_fft  # ✏️ TODO: 每个 bin 的频率

    chroma = np.zeros(12)
    for k, (f, p) in enumerate(zip(freqs, power_spectrum)):
        if f <= 0:
            continue
        # ✏️ TODO: 计算 midi = 12 * log2(f/440) + 69
        midi = 0.0
        # ✏️ TODO: pc = round(midi) mod 12，累加到 chroma[pc]
        pass

    # ✏️ TODO: 归一化
    return chroma

In [ ]:
# 检查：纯 A4 信号的 chroma[9] 应最高
sr_c, n_fft_c = 22050, 2048
t_c = np.arange(n_fft_c) / sr_c
x_c = np.sin(2 * np.pi * 440 * t_c) * np.hanning(n_fft_c)
pspec = np.abs(np.fft.rfft(x_c)) ** 2

ch = chroma_vector(pspec, sr_c, n_fft_c)
peak_pc = ch.argmax()
assert peak_pc == 9, f'期望 9 (A), 得到 {peak_pc}'
assert abs(ch[9] - 1.0) < 1e-6, '最高值应为 1.0（归一化）'
print(f'✅ chroma_vector OK：peak pitch class = {peak_pc} (A)，chroma[9] = {ch[9]:.4f}')

## 5. ✏️ 实现 `rms_envelope(x, frame_len=2048, hop=512)`

**签名**：`rms_envelope(x, frame_len=2048, hop=512) -> np.ndarray  # shape (n_frames,)`

**推理路线**：
1. 计算帧数：`n_frames = 1 + (len(x) - frame_len) // hop`
2. 用列表推导或 stride trick 切出每帧：`frames[i] = x[i*hop : i*hop+frame_len]`
3. 对每帧计算 `sqrt(mean(frame**2))`，得到长度 `n_frames` 的数组

**参考输入输出**：
- 输入：`x = np.zeros(10000)`（纯静音）→ 输出：全 0 数组
- 输入：`x = np.ones(10000) * 0.5` → 输出：每帧 RMS ≈ 0.5

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def rms_envelope(x, frame_len=2048, hop=512):
    """
    计算音频信号的逐帧 RMS 能量包络。

    Parameters
    ----------
    x : np.ndarray, shape (N,)  单声道音频
    frame_len : int  帧长（采样点数）
    hop : int  跳步

    Returns
    -------
    rms : np.ndarray, shape (n_frames,)
    """
    # ✏️ TODO: 计算 n_frames = 1 + (len(x) - frame_len) // hop
    # ✏️ TODO: 切帧 frames[i] = x[i*hop : i*hop+frame_len]，shape = (n_frames, frame_len)
    # ✏️ TODO: 返回每帧的 sqrt(mean(frame**2))
    raise NotImplementedError("rms_envelope: 请完成实现 — 参考推理路线三步")

In [ ]:
# 检查 rms_envelope
try:
    # 静音 → RMS ≈ 0
    x_sil = np.zeros(10000)
    rms_sil = rms_envelope(x_sil)
    assert rms_sil.max() < 1e-10, f'静音 RMS 应约为 0，得 {rms_sil.max()}'

    # 常数信号 0.5 → RMS ≈ 0.5
    x_const = np.ones(10000) * 0.5
    rms_const = rms_envelope(x_const)
    assert np.allclose(rms_const, 0.5, atol=1e-6), f'常数信号 RMS 应为 0.5，得 {rms_const.mean():.4f}'

    print(f'✅ rms_envelope OK：静音 max={rms_sil.max():.2e}，常数 mean={rms_const.mean():.4f}')
except NotImplementedError as e:
    print(f'⏸ rms_envelope 尚未实现：{e}\n请按推理路线完成 cell 13 中的 TODO。')

## 6. 参数实验：hop 对 RMS 时间分辨率的影响

改变 `hop` 参数，观察 RMS 包络的时间分辨率与平滑程度：

- `hop=128`：高时间分辨率，包络抖动，能捕捉到快速瞬态（< 6 ms）
- `hop=512`（默认）：均衡，约 23 ms 步长，适合节拍检测
- `hop=2048`：低分辨率，包络极平滑，接近整体响度，节拍细节丢失

预期现象：`hop` 越大，曲线越平滑，节拍峰越模糊；`hop` 越小，曲线越细腻，但计算量正比增大。

In [ ]:
# 参数实验：不同 hop 的 RMS 包络对比
sr_exp = 22050
bpm_exp = 120
beat_period = int(60 / bpm_exp * sr_exp)
x_exp = np.zeros(sr_exp * 4)
for bs in range(0, len(x_exp), beat_period):
    end = min(bs + 2205, len(x_exp))
    length = end - bs
    x_exp[bs:end] += np.exp(-np.arange(length) / 800) * 0.9

frame_len_exp = 2048
hops = [128, 512, 2048]
colors = ['tab:blue', 'tab:orange', 'tab:green']

plt.figure(figsize=(11, 4))
for h, c in zip(hops, colors):
    rms_e = rms_envelope(x_exp, frame_len=frame_len_exp, hop=h)
    t_e = np.arange(len(rms_e)) * h / sr_exp
    plt.plot(t_e, rms_e, label=f'hop={h}', color=c, alpha=0.8)

plt.xlabel('时间 (s)')
plt.ylabel('RMS')
plt.title('不同 hop 的 RMS 包络（120 BPM 模拟击鼓）')
plt.legend()
plt.tight_layout()
plt.show()
print('观察：hop=128 最细腻，hop=2048 最平滑；节拍峰在小 hop 时更清晰。')

## 7. ✏️ 实现 `zero_crossing_rate(x, frame_len=2048, hop=512)`

**签名**：`zero_crossing_rate(x, frame_len, hop) -> np.ndarray  # shape (n_frames,)`

**原理**：ZCR（过零率）统计音频帧内信号从正变负（或从负变正）的次数，归一化到帧长：

```
zcr[i] = sum( |sign(frame[n]) - sign(frame[n-1])| ) / (2 * frame_len)
```

纯正弦波 ZCR ≈ `2f/sr`（频率越高过零越频繁）；白噪声 ZCR ≈ 0.5（随机正负号）。

**推理路线**：
1. 复用 `rms_envelope` 的分帧逻辑（n_frames、切帧）
2. 对每帧计算 `np.diff(np.sign(frame))` 非零元素个数
3. 除以 `2 * frame_len` 归一化

> ⚠️ `np.sign(0) = 0`，两侧都非零才算一次过零；用 `np.sign` 而非直接比较可自动处理零点。

> 💡 完成后可与 `src/aurora/music/features.zero_crossing_rate` 对比（该函数同样逐帧计算，归一化方式相同）。


In [ ]:
def zero_crossing_rate(x, frame_len=2048, hop=512):
    """
    计算音频信号的逐帧过零率（Zero Crossing Rate）。

    Parameters
    ----------
    x : np.ndarray, shape (N,)
    frame_len : int
    hop : int

    Returns
    -------
    zcr : np.ndarray, shape (n_frames,)  每帧过零率，范围 [0, 0.5]
    """
    # ✏️ TODO: 1. 计算 n_frames（与 rms_envelope 相同公式）
    # ✏️ TODO: 2. 切帧
    # ✏️ TODO: 3. 对每帧 np.diff(np.sign(frame)) 统计非零数 / (2 * frame_len)
    raise NotImplementedError("zero_crossing_rate: 请完成实现")


In [ ]:
# 检查 zero_crossing_rate
try:
    sr_z = 22050
    # 纯 A4 正弦波：ZCR ≈ 2 * 440 / 22050 ≈ 0.0399
    t_z = np.arange(sr_z) / sr_z
    x_sine = np.sin(2 * np.pi * 440 * t_z)
    zcr_sine = zero_crossing_rate(x_sine, frame_len=2048, hop=512)
    expected_zcr = 2 * 440 / sr_z   # ≈ 0.0399
    assert np.allclose(zcr_sine.mean(), expected_zcr, atol=0.005), \
        f'正弦 ZCR 应约为 {expected_zcr:.4f}，得 {zcr_sine.mean():.4f}'

    # 白噪声：ZCR ≈ 0.5（随机正负号，过零极频繁）
    rng = np.random.default_rng(42)
    x_noise = rng.standard_normal(sr_z)
    zcr_noise = zero_crossing_rate(x_noise, frame_len=2048, hop=512)
    assert zcr_noise.mean() > 0.45, f'白噪声 ZCR 应 > 0.45，得 {zcr_noise.mean():.4f}'

    print(f'✅ zero_crossing_rate OK：正弦 ZCR mean={zcr_sine.mean():.4f}（期望≈{expected_zcr:.4f}），'
          f'噪声 ZCR mean={zcr_noise.mean():.4f}（期望≈0.5）')
except NotImplementedError as e:
    print(f'⏸ zero_crossing_rate 尚未实现：{e}\n请按推理路线完成上方 stub。')


## 本课收束

本节实现了 `chroma_vector`（12维调性指纹，输出归一化 ndarray）和 `rms_envelope`（逐帧均方根，输出响度时间序列），并演示了自相关节拍估计（BPM 定位）——三者将共同构成 `aurora.music` 特征提取管线的核心。下一节（**L78**）将从零实现节拍追踪：计算 onset 包络、用自相关估计 BPM，并在合成鼓点信号上验证精度。